# Exploratory Data Analysis for Pipe Break Prediction

## 1. Introduction
The objective of our project is to predict the age at which a water main might break. By doing so, we hope to enable preventive maintenance and significantly reduce the impact of water main breaks on our communities.Our dataset comprises historical water main break incidents and various associated infrastructure data.

By analyzing this historical data, we aim to develop a robust machine learning model that can accurately identify potential high-risk water mains before they break. The capability to predict these breakages in advance not only has the potential to improve the longevity of our water infrastructure but also provides an opportunity to manage resources more efficiently and prevent disruptions.

This brings us to our Exploratory Data Analysis (EDA) notebook. The goal of this notebook is to understand the data that we have in detail. During EDA, we inspect the structure of the data, look for potential outliers or anomalies, identify patterns, and discover relationships among variables.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff

# load in the data
data = pd.read_csv("../data/interim/cleaned_break_data.csv")
print(data.shape)
data.head(15)

In [ ]:
# missing values for each column
data.isnull().sum()

In [ ]:
# making the columns lowercase
data.columns = data.columns.str.lower()

In [ ]:
data.info()

Looking at the head of the data that we called, and after further inspection it seems that there is an occurrence where the incidents are duplicating, some have different installation dates, some have different criticality scores, as well as different condition scores. This is an interesting observation, and I'm afraid now that all of those duplicates will have to be removed at some point to be able to get a fully unique dataset. We can see how many actual unique cases there are in the data after running the command below. I will carry on for now as if this isn't occurring and push through the EDA, and then hopefully figure out how to take care of the situation later.

In [ ]:
data['incident_date'].nunique()

Target variable creation

In [ ]:
data['incident_date'] = pd.to_datetime(data['incident_date'])
data['installation_date'] = pd.to_datetime(data['installation_date'])

# Calculate the age of the pipe at the time of breakage
data['age_at_break'] = (data['incident_date'].dt.year - data['installation_date'].dt.year)

# Now, for frequency of breaks, first, let's calculate the operational years of each pipe
data['operational_years'] = (data['incident_date'].max().year - data['installation_date'].dt.year)

# Create a dataframe counting the number of breaks per pipe
break_counts = data.groupby('assetid')['incident_date'].count().reset_index()
break_counts.columns = ['assetid', 'num_breaks']

# Add the operational years to the break_counts dataframe
break_counts = break_counts.merge(data[['assetid', 'operational_years']].drop_duplicates(), on='assetid', how='left')

# Calculate the frequency of breaks
break_counts['breaks_per_year'] = break_counts['num_breaks'] / break_counts['operational_years']

# Merge this back into the original dataframe
data = data.merge(break_counts, on='assetid', how='left')

In [ ]:
data.sample(5)

In [ ]:
# Plotting age_at_break
plt.figure(figsize=(10, 6))
sns.histplot(data=data, x="age_at_break", bins=30, kde=True)
plt.title("Distribution of Age at Break")
plt.show();

In [ ]:
# Plotting breaks_per_year
plt.figure(figsize=(10, 6))
sns.histplot(data=data, x="breaks_per_year", bins=30, kde=True)
plt.title("Distribution of Breaks Per Year")
plt.show();

In [ ]:
# Compute the correlation matrix
corr = data.corr()

# Generate a mask for the upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))

# Set up the matplotlib figure
f, ax = plt.subplots(figsize=(11, 9))

# Generate a custom diverging colormap
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(corr, mask=mask, cmap=cmap, vmax=.3, center=0,
            square=True, linewidths=.5, cbar_kws={"shrink": .5});

In [ ]:
fig = px.histogram(data, x='hour_impacted', marginal="box")
fig.show();

In [ ]:
data = data.replace([np.inf, -np.inf], np.nan)
data = data.dropna()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=data, x="asset_material", y="num_breaks")
plt.title("Distribution of Breaks Per Year by Pipe Material")
plt.show();

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=data, x="asset_size", y="num_breaks")
plt.title("Distribution of Breaks Per Year by Asset Size")
plt.show();

In [ ]:
fig = px.scatter(data, x="breaks_per_year", y="asset_size", color="num_breaks",
                    size='num_breaks', hover_data=['assetid'], opacity=0.01)
fig.show();

In [ ]:
# pie chart of pipe material
fig = px.pie(data, values='num_breaks', names='asset_material', title='Breaks by Pipe Material')
fig.show();

In [ ]:
# condition score base on pipe material
fig = px.box(data, x="asset_material", y="condition_score", color="asset_material", title="Condition Score by Pipe Material")
fig.show();

In [ ]:
# distribution of condition score with seaborn
plt.figure(figsize=(10, 6))
sns.histplot(data=data, x="condition_score", bins=30, kde=True)
plt.title("Distribution of Condition Score")
plt.show();

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=data, x="asset_size", y="condition_score")
plt.title("Distribution of Condition Score by Asset Size")
plt.show();

In [ ]:
# distribution of asset year installed
plt.figure(figsize=(10, 6))
sns.histplot(data=data, x="installation_date", bins=30, kde=True)
plt.title("Distribution of Asset Year Installed")
plt.show();

In [ ]:
fig = px.histogram(data, x="installation_date", marginal="violin", title="Distrbution of Asset Year Installed")
fig.show();

In [ ]:
# criticality score
fig = px.histogram(data, x="criticality", title="Distribution of Criticality Score")
fig.show();

In [ ]:
data.columns

In [ ]:
# distribution of operational years
plt.figure(figsize=(10, 6))
sns.histplot(data=data, x="operational_years_x", bins=30, kde=True)
plt.title("Distribution of Operational Years")
plt.show();

In [ ]:
fig = px.scatter_mapbox(data, lat="latitude", lon="longitude", hover_name="assetid", hover_data=["assetid", "criticality"],
                        color_discrete_sequence=["fuchsia"], zoom=10, height=300)
fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(margin={"r": 0, "t": 0, "l": 0, "b": 0})
fig.show();

In [ ]:
fig = px.density_mapbox(data, lat='latitude', lon='longitude', z='num_breaks', radius=10,
                        center=dict(lat=43.45, lon=-80.5), zoom=9,
                        mapbox_style="stamen-terrain")
fig.show();

In [ ]:
fig = px.scatter_mapbox(data, lat="latitude", lon="longitude", hover_name="assetid", hover_data=["assetid", "condition_score"],
                        color_discrete_sequence=["fuchsia"], zoom=10, height=300, size='num_breaks', animation_frame=data['incident_date'].dt.year.sort_values())
fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(margin={"r": 0, "t": 0, "l": 0, "b": 0})
fig.show();

In [ ]:
data_temp = data.set_index('incident_date')
data_temp.resample('Y').count()['assetid'].plot(figsize=(12, 8))

In [ ]:
fig = px.line(data_temp.resample('Y').count()['num_breaks'], title='Number of Breaks Per Year')
fig.show();

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
breaks_per_month = data_temp.resample('M').count()["num_breaks"]
decomposition = seasonal_decompose(breaks_per_month)
fig = plt.figure()
fig = decomposition.plot()
fig.set_size_inches(15, 8);

In [ ]:
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(breaks_per_month)

In [ ]:
data['objectid'].nunique()

In [ ]:
# show the data points where the age is negative
data[data['age_at_break'] < 0]

In [ ]:
# show the data points where the age is negative and show the incident date and installation date and match the objectid using groupby
negative_age_df = data[data['age_at_break'] < 0]

cols = ['objectid', 'incident_date', 'installation_date']
negative_age_df = negative_age_df[cols]
negative_age_df.head(10)

In [ ]:
negative_age_df = data[data['age_at_break'] < 0]

cols = ['objectid', 'asset_year_installed', 'incident_date', 'installation_date']
negative_age_df = negative_age_df[cols]
print(negative_age_df.shape)
negative_age_df.head(10)